In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import text

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from cryptoquant.database.session import get_session
from cryptoquant.config.settings import get_settings

print("✓ Imports successful")
print(f"Current time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")

In [ ]:
session = get_session()
settings = get_settings()
print(f"✓ Connected to: {settings.database_url}")

In [ ]:
# Detect ALL gaps with specific timestamps
query = text("""
WITH ordered_candles AS (
    SELECT
        mp.trading_pair_id,
        tp.symbol,
        mp.timestamp AS gap_start,
        LEAD(mp.timestamp) OVER (PARTITION BY mp.trading_pair_id ORDER BY mp.timestamp) AS gap_end,
        DATEDIFF(HOUR, mp.timestamp, LEAD(mp.timestamp) OVER (PARTITION BY mp.trading_pair_id ORDER BY mp.timestamp)) AS hours_gap
    FROM crypto.market_prices AS mp
    INNER JOIN crypto.trading_pairs AS tp ON tp.id = mp.trading_pair_id
)
SELECT
    symbol AS currency_pair,
    gap_start,
    gap_end,
    hours_gap,
    DATEADD(HOUR, 1, gap_start) AS backfill_start,
    DATEADD(HOUR, -1, gap_end) AS backfill_end
FROM ordered_candles
WHERE gap_end IS NOT NULL
  AND hours_gap > 1
ORDER BY symbol, gap_start
""")

df_gaps = pd.DataFrame(session.execute(query).fetchall(),
                       columns=['currency_pair', 'gap_start', 'gap_end', 'hours_gap',
                               'backfill_start', 'backfill_end'])

print(f"=== Detected {len(df_gaps)} Gap(s) ===")
display(df_gaps)

if len(df_gaps) > 0:
    total_hours = df_gaps['hours_gap'].sum()
    print(f"\n⚠️  Total missing hours: {total_hours:,.0f}")
    print(f"⚠️  Pairs affected: {df_gaps['currency_pair'].nunique()}")
else:
    print("\n✓ No gaps detected!")

In [ ]:
# Summary by pair
if len(df_gaps) > 0:
    summary = df_gaps.groupby('currency_pair').agg({
        'hours_gap': ['count', 'sum', 'mean', 'max']
    }).round(2)
    summary.columns = ['gap_count', 'total_hours_missing', 'avg_gap_hours', 'max_gap_hours']
    summary = summary.sort_values('total_hours_missing', ascending=False)
    
    print("\n=== Gap Summary by Pair ===")
    display(summary)

In [ ]:
# Generate Python backfill code
print("\n=== Backfill Code (Copy to next cell or Python script) ===\n")
print("from datetime import datetime, timezone")
print("from cryptoquant.ingestion.backfill import backfill_multiple_gaps")
print("import logging\n")
print("logging.basicConfig(level=logging.INFO)\n")
print("gaps_to_fill = [")

for idx, row in df_gaps.iterrows():
    pair = row['currency_pair']
    start = row['backfill_start']
    end = row['backfill_end']
    hours = row['hours_gap']
    
    # Format as Python datetime
    start_str = f"datetime({start.year}, {start.month}, {start.day}, {start.hour}, 0, 0, tzinfo=timezone.utc)"
    end_str = f"datetime({end.year}, {end.month}, {end.day}, {end.hour}, 0, 0, tzinfo=timezone.utc)"
    
    print(f"    {{")
    print(f"        'product_id': '{pair}',")
    print(f"        'start_date': {start_str},")
    print(f"        'end_date': {end_str},")
    print(f"        # Gap: {hours} hours")
    print(f"    }},")

print("]\n")
print("# Execute backfill")
print("stats = backfill_multiple_gaps(gaps_to_fill, granularity='hourly')")
print("print(f'\\n✅ Backfill complete: {stats}')")

In [ ]:
# Execute backfill (uncomment to run)
# NOTE: Paste the generated code from above or run it manually

# Example for a single gap:
# from cryptoquant.ingestion.backfill import backfill_candle_gap
# stats = backfill_candle_gap(
#     product_id='BTC-USD',
#     start_date=datetime(2026, 8, 1, 10, 0, 0, tzinfo=timezone.utc),
#     end_date=datetime(2026, 8, 1, 15, 0, 0, tzinfo=timezone.utc),
#     granularity='hourly'
# )
# print(f'✅ Backfill complete: {stats}')

print("⚠️  Backfill code ready. Copy from cell above to execute.")

In [ ]:
session.close()
print("✓ Database connection closed")